<a href="https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/fix/OilSlick_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install rasterio captum terratorch

In [ ]:
import os
import sys
from getpass import getpass

# Securely input GitHub Personal Access Token
github_token = getpass('Enter your GitHub Personal Access Token (ghp_...): ')

repo_url = f"https://{github_token}@github.com/astronerd21/Deep-Learning-Oil-Slick-Detection.git"

%cd /content
!rm -rf Deep-Learning-Oil-Slick-Detection

# Clone the specific development branch containing the preprocessing logic
!git clone {repo_url}

sys.path.append('/content/Deep-Learning-Oil-Slick-Detection')

# Verify that the custom dataset and model architecture modules load correctly
try:
    from src.dataset import SARDataset
    from src.model import SARResNet
    print("Success! GitHub repository code and modules are ready.")
except ImportError as e:
    print(f"Error loading modules: {e}")

In [ ]:
import os

# Define path configurations for Drive storage and local scratch space
drive_dir = '/content/drive/MyDrive/OilSlickProject/data/OilSlick/'
local_temp_dir = '/content/temp_archives/'
local_base_dir = '/content/data/'
local_images_dir = '/content/data/images_s1'

os.makedirs(local_temp_dir, exist_ok=True)
os.makedirs(local_base_dir, exist_ok=True)
os.makedirs(local_images_dir, exist_ok=True)

# Verify if the Sentinel-1 image chips are already extracted locally
image_count = len(os.listdir(local_images_dir))

if image_count < 980:
    print("Step 1/3: Copying 7.5 GB tar archives from Google Drive to local runtime storage...")
    !cp {drive_dir}OilSlick-images_s1-00.tar {local_temp_dir}
    !cp {drive_dir}OilSlick-images_s1-01.tar {local_temp_dir}

    print("Step 2/3: Extracting radar image chips...")
    !tar -xf {local_temp_dir}OilSlick-images_s1-00.tar -C {local_base_dir}
    !tar -xf {local_temp_dir}OilSlick-images_s1-01.tar -C {local_base_dir}

    print("Step 3/3: Cleaning up temporary archive files...")
    !rm -rf {local_temp_dir}

    print("\nLocal data environment setup complete!")
else:
    print("Dataset already unpacked and available locally.")

print(f"Current total number of S1 image chips on disk: {len(os.listdir(local_images_dir))}")

In [ ]:
%cd /content/Deep-Learning-Oil-Slick-Detection

print("=== Starting Dataset Cleaning ===")
# Execute script to clean splits based on the 50% NoData threshold
!PYTHONPATH=. python3 -m src.clean_dataset \
    --data-dir "/content/data/images_s1" \
    --splits-in-dir "/content/drive/MyDrive/OilSlickProject/data/OilSlick/splits/random"

In [ ]:
import os
import random
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import copy

# 1. Path configuration
img_dir = '/content/data/images_s1/'
clean_split_path = '/content/cleaned_splits/train_clean.txt'

if not os.path.exists(clean_split_path):
    raise FileNotFoundError(f"Cleaned split file missing at {clean_split_path}. Run the cleaning step first.")

# Load verified IDs that passed the NoData filter check
with open(clean_split_path, 'r') as f:
    valid_ids = [line.strip() for line in f if line.strip()]

# Sample 5 random valid image IDs
sample_ids = random.sample(valid_ids, 5)

def stretch_contrast_robust(band):
    """Native contrast stretch using NaN for safe NoData masking."""
    b = band.copy()

    # 1. Convert exact 0.0 NoData values to NaN (Not a Number)
    # This forces matplotlib and numpy to completely ignore these pixels mathematically
    b[b == 0.0] = np.nan

    # If the image is somehow completely empty, return as is
    if np.isnan(b).all():
        return b

    # 2. Calculate the 2nd and 98th percentiles strictly on the valid sea surface
    vmin, vmax = np.nanpercentile(b, (2, 98))

    # 3. Clip the values to these bounds (NaNs remain intact!)
    return np.clip(b, vmin, vmax)


# 2. Build the visual gallery
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Configure the colormap to paint all NaN values as pitch black
cmap = copy.copy(plt.cm.gray)
cmap.set_bad(color='black')

for i, sample_id in enumerate(sample_ids):
    file_name = f"{sample_id}_s1.tif" if not sample_id.endswith('.tif') else sample_id
    img_path = os.path.join(img_dir, file_name)
    display_id = sample_id.replace('_s1.tif', '')

    is_oil = "OIL SLICK (1)" if (display_id.startswith('pos') or display_id.startswith('ext_pos')) else "NO OIL (0)"

    # Open raw GeoTIFF directly from disk
    with rasterio.open(img_path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)

    # Apply the robust transformation
    vv_vis = stretch_contrast_robust(vv)
    vh_vis = stretch_contrast_robust(vh)

    # Top row: VV Channel
    # We let matplotlib handle the scaling natively using our NaN-aware colormap
    ax_vv = axes[0, i]
    ax_vv.imshow(vv_vis, cmap=cmap)
    ax_vv.set_title(f"VV Channel | {display_id}\nLabel: {is_oil}", fontsize=11)
    ax_vv.axis('off')

    # Bottom row: VH Channel
    ax_vh = axes[1, i]
    ax_vh.imshow(vh_vis, cmap=cmap)
    ax_vh.set_title(f"VH Channel | {display_id}\nLabel: {is_oil}", fontsize=11)
    ax_vh.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
%cd /content/Deep-Learning-Oil-Slick-Detection

print("=== Starting Model Training (TerraMind Foundation Model) ===")
# Note: Batch size is reduced to 16 (from 64) due to the large memory footprint of the ViT backbone.
!PYTHONPATH=. python3 -m src.train \
    --data-dir "/content/data/images_s1" \
    --train-split "/content/cleaned_splits/train_clean.txt" \
    --val-split-file "/content/cleaned_splits/val_clean.txt" \
    --backbone terramind \
    --epochs 20 \
    --batch-size 16 \
    --output "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_terramind_model.pt"

In [ ]:
%cd /content/Deep-Learning-Oil-Slick-Detection

print("=== Starting Model Training ===")
# Launch training using the absolute path of the generated clean split text files
!PYTHONPATH=. python3 -m src.train \
    --data-dir "/content/data/images_s1" \
    --train-split "/content/cleaned_splits/train_clean.txt" \
    --val-split-file "/content/cleaned_splits/val_clean.txt" \
    --epochs 20 \
    --batch-size 64 \
    --output "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt"

In [ ]:
# Cell: Grad-CAM Explainability Visualization (Multi-Sample Grid)
import torch
import rasterio
import random
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
from src.model import SARResNet
from src.xai import GradCAM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt"
model = SARResNet(backbone="resnet18", pretrained=False).to(device)

try:
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint.get("model_state_dict", checkpoint))
    print("Model weights successfully loaded.")
except Exception as e:
    print(f"Error loading model: {e}")

model.eval()

with open('/content/cleaned_splits/train_clean.txt', 'r') as f:
    valid_ids = [line.strip() for line in f if line.strip()]

# Filter for IDs containing oil slicks (starting with 'pos' or 'ext_pos')
tp_ids = [vid for vid in valid_ids if vid.startswith('pos') or vid.startswith('ext_pos')]

# Number of samples for the grid
num_samples = 3
sample_ids = random.sample(tp_ids, num_samples)

# Prepare transformation and Grad-CAM
transform = T.Compose([T.Resize((224, 224), antialias=True)])
target_layer = model.net.layer4[-1]
grad_cam = GradCAM(model, target_layer)

# Build the grid visualization
fig, axes = plt.subplots(num_samples, 3, figsize=(16, 5 * num_samples))
fig.suptitle("Grad-CAM Analysis: Stability of Feature Extraction (Wave Damping)", fontsize=16, y=1.02)

for i, sample_id in enumerate(sample_ids):
    img_path = f'/content/data/images_s1/{sample_id}'
    if not img_path.endswith('.tif'):
        img_path += '_s1.tif'

    # Load data and normalize as in the dataset
    with rasterio.open(img_path) as src:
        data = src.read().astype(np.float32)

    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std

    tensor_img = torch.from_numpy(data)
    input_tensor = transform(tensor_img).unsqueeze(0).to(device)

    # Generate heatmap
    heatmap = grad_cam.generate_heatmap(input_tensor)

    # Extract VV channel
    vv_vis = input_tensor[0, 0].cpu().numpy()

    # --- Plot the 3 columns per row ---

    # Column 1: Raw VV
    axes[i, 0].imshow(vv_vis, cmap='gray')
    axes[i, 0].set_title(f"VV Channel\nID: {sample_id}")
    axes[i, 0].axis('off')

    # Column 2: Pure Heatmap
    axes[i, 1].imshow(heatmap, cmap='jet')
    axes[i, 1].set_title("Grad-CAM Heatmap")
    axes[i, 1].axis('off')

    # Column 3: Overlay
    axes[i, 2].imshow(vv_vis, cmap='gray')
    axes[i, 2].imshow(heatmap, cmap='jet', alpha=0.45)
    axes[i, 2].set_title("Network Focus Overlay")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Das Evaluierungs-Skript auf euren ResNet-Checkpoint loslassen!
!PYTHONPATH=. python3 -m src.evaluate \
    --data-dir "/content/data/images_s1" \
    --model "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_cleaned_model.pt" \
    --split "/content/cleaned_splits/val_clean.txt" \
    --backbone resnet18

In [ ]:
# Das Evaluierungs-Skript auf euren TerraMind-Checkpoint loslassen!
!PYTHONPATH=. python3 -m src.evaluate \
    --data-dir "/content/data/images_s1" \
    --model "/content/drive/MyDrive/OilSlickProject/data/OilSlick/checkpoints/best_terramind_model.pt" \
    --split "/content/cleaned_splits/val_clean.txt" \
    --backbone terramind \
    --batch-size 16